# NewsCheck TN — Lot C : avancement Model 1

**Tâche :** détecter les phrases contenant une affirmation vérifiable.  
**Classes :** `AFFIRMATION`, `OPINION`, `AUTRE`.  
**Périmètre :** ce modèle ne décide pas si une affirmation est vraie ou fausse. Le verdict dépend des preuves, du NLI et des étapes du Lot D.

> Ce notebook présente les résultats communiqués pour les expériences déjà effectuées. Les tableaux ci-dessous sont documentés en Markdown : ils ne sont pas recalculés automatiquement à l'ouverture. Les cellules de vérification en fin de notebook sont facultatives et nécessitent les fichiers du projet.

## 1. Données et protocole

- 600 phrases annotées : 316 `AFFIRMATION`, 93 `OPINION`, 191 `AUTRE`.
- Langues : 160 FR, 150 AR et 290 TN ; metadata de langue complétées pour 60 phrases initialement `unk`.
- Un test de 120 identifiants reste identique pour toutes les comparaisons : FR 26, AR 30, TN 64.
- XLM-R : pour chaque seed, 408 exemples d'entraînement et 72 de validation ; le partage train/validation varie selon la seed.
- Les baselines utilisent les 480 exemples hors test pour l'entraînement.

**À confirmer avec l'équipe :** alignement des données et du test gelé, qualité des annotations et droits de partage des textes avant diffusion.

## 2. Modèles comparés

1. **Baseline classique :** TF-IDF de caractères + régression logistique.
2. **Baseline neuronale :** embeddings MiniLM multilingues figés + régression logistique.
3. **Modèle avancé :** `FacebookAI/xlm-roberta-base` adapté avec PEFT / LoRA, sur 3 seeds : 42, 123 et 2026.

LoRA entraîne des adaptateurs et la tête de classification, sans réentraîner tous les poids de XLM-R. Le checkpoint d'intégration utilisé ici est `seed42`.

## 3. Résultats sur le test gelé

Macro-F1 : moyenne ± écart-type des runs indiqués. Les trois résultats des baselines sont identiques ; cela ne constitue pas trois jeux de test indépendants.

| Modèle | Global | FR (n=26) | AR (n=30) | TN (n=64) |
| --- | ---: | ---: | ---: | ---: |
| TF-IDF + LR | 0,6435 ± 0,0000 | 0,5760 | 0,4778 | 0,6846 |
| MiniLM figé + LR | 0,6350 ± 0,0000 | 0,8228 | 0,8002 | 0,4886 |
| XLM-R + LoRA | **0,7357 ± 0,0317** | 0,9148 ± 0,0000 | 0,6882 ± 0,0185 | 0,6863 ± 0,0567 |

XLM-R par seed : 42 = 0,7108 ; 123 = 0,7715 ; 2026 = 0,7250. Les sous-groupes linguistiques sont petits : interprétation prudente.

## 4. Bootstrap, ablation et analyse d'erreurs

**Bootstrap apparié**, TF-IDF vs XLM-R seed42, 1 000 réplications : différence de macro-F1 = **+0,0673**, IC 95 % **[−0,0468 ; 0,1860]**, p = **0,254**. L'intervalle contient zéro : l'amélioration observée n'est pas statistiquement concluante sur ce test de 120 phrases.

**Ablation raw vs normalisé**, seed42 : raw = **0,7108**, normalisé = **0,6671** ; différence normalisé − raw = −0,0436, IC 95 % [−0,1184 ; 0,0202], p = 0,188. Le raw est retenu, sans conclure que l'écart est significatif.

**Erreurs :** 32 identifiants incorrects au moins une fois sur trois seeds, dont 18 incorrects pour les trois. Une analyse qualitative de 30 cas est disponible dans `docs/lot-c_error_analysis_results.md`.

## 5. Intégration et limites

- `POST /extract` et `POST /check` ont été essayés manuellement en local avec succès.
- Le détecteur utilise le checkpoint `seed42` lorsqu'il est disponible ; sinon un fallback heuristique est signalé dans `/health`.
- L'extraction et les étapes aval utilisent le texte normalisé ; une extraction parallèle fournit les phrases brutes à Model 1. Si le nombre de candidats diffère, le pipeline utilise le texte normalisé en repli. Cet alignement par ordre doit encore être évalué pour des textes complexes.
- L'abstention `preuves_insuffisantes` ne signifie pas que l'affirmation est fausse.
- Model 2 NLI fine-tuné reste à réaliser après réception et validation de paires annotées `claim / evidence / supports-refutes-neutral`.
- Le temps de training des trois checkpoints historiques n'a pas été conservé ; les temps de test ne sont pas des temps de training.

## 6. Environnement de référence

Capture communiquée : Google Colab, GPU Tesla T4, Python 3.13.16, torch 2.11.0+cu130, transformers 5.20.2, peft 0.21.2, accelerate 1.15.0, datasets 4.8.5, sentence-transformers 5.7.0, scikit-learn 1.6.1 et numpy 2.1.3. La capture décrit l'environnement au moment du relevé ; elle ne garantit pas à elle seule que toutes ces versions étaient identiques pour chaque ancien run.

**Important :** checkpoints, corpus bruts et sorties d'évaluation sont exclus par `.gitignore`. Le notebook seul ne contient donc ni les adaptateurs ni toutes les prédictions. Vérifier les droits avant de partager les données ou modèles.

## 7. Vérification facultative des données

Ouvrir ce notebook depuis la racine de la copie du dépôt, ou placer le projet dans `/content/NewsCheck` sur Colab. La cellule suivante ne télécharge ni n'entraîne aucun modèle. Si les fichiers ne sont pas présents, les tableaux ci-dessus restent lisibles.

In [ ]:
from pathlib import Path
import json
from collections import Counter

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/NewsCheck')]
root = next((p for p in candidates if (p / 'data/processed/claim_detection_600.jsonl').is_file()), None)
if root is None:
    print('Dataset introuvable : ouvrir le notebook depuis une copie complète du projet.')
else:
    data = root / 'data/processed/claim_detection_600.jsonl'
    rows = [json.loads(line) for line in data.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('Racine :', root)
    print('Nombre de phrases :', len(rows))
    print('Labels :', dict(Counter(row.get('label') for row in rows)))
    print('Langues :', dict(Counter(row.get('lang') for row in rows)))
    print('Split gelé présent :', (root / 'data/splits/claim_detection_multiclass_600_seed42.json').is_file())
    print('Checkpoint seed42 présent :', (root / 'models/checkpoints/claim_detection_xlmr_lora_seed42').is_dir())

## 8. Pour aller plus loin

- Rapport détaillé : `docs/model1_claim_detection_detailed_report.md`.
- Model Card : `models/model_cards/claim_detection_xlmr_lora.md`.
- Reproductibilité : `docs/model1_reproducibility.md`.
- Journal : `experiments.md`, expériences Lot C.
- Explication pédagogique en arabe : `docs/explication_arabe_model1_et_pipeline.md`.

Les scripts d'entraînement et d'évaluation sont sous `models/encoders/` ; les scripts de baselines et de bootstrap sont sous `models/baselines/` et `models/evaluation/`.